# LO benchmark results

Load multi-model results for the linear-optimization (LO / LP) benchmark into one dataframe, then show a **score table**: rows = questions, columns = LLMs.

Each cell is `1` if the model's JSON `cost` was within 1% of the keyed objective, else `0`.

**Download in this notebook:** the load cell sets `DOWNLOAD_KAGGLE_RUNS = True` and pulls runs via the Kaggle CLI into `data/kaggle_runs/lo-normative-accuracy-2`. Requires `kaggle auth login` once.

If download is off and files are missing, the cell will still try to download automatically.

Or from a local sandbox / merged CSV: set `LOAD_FROM_KAGGLE = False` and put `*.run.json` or `*merged*.csv` under `data/lp/` or one of the sandbox candidate folders.

In [3]:
! \src\projects\sceptical_llms\.venv\Scripts\python.exe -m kaggle auth login

Your browser has been opened to visit:
  https://www.kaggle.com/api/v1/oauth2/authorize?response_type=code&client_id=kagglesdk&redirect_uri=http%3A%2F%2Flocalhost%3A8090&scope=resources.admin%3A%2A&state=e259ad30-f7dd-433b-aeee-11d4afda2f49&code_challenge=YVWRcHY2cshRp4HjH9Nmp5tNhIlvkinvcECXcELVh4I%3D&code_challenge_method=S256&response_mode=query



You are now logged in as [francisganong]



127.0.0.1 - - [20/Jul/2026 20:41:26] "GET /?code=CfDJ8HMxIwPZ_4ZImnEqAVaKkSFEpaR03Gnt2jaF_WySqHOS8M5Dny8f7PzxIT11a6I3aZ_Ga8BJYBgiUpswryAnC7_-0t4Y3Ob3ThwnHCp1-Qem6XisvNlPYX3tIrTyEe0PkEm1TJfPjiWIkTnpI-IQu9u5O6tklq2Q9sA3l88mVZX0srsBz73f7Dmf0fbsHm30M9_HhKp1LdDRSHh01N16cDgKlEG8E8ZMyEGR8yEcrzbqcE4NyGdYFCPwRGscuMFdBqzMCAlIz_Kz8SSNVBBDlpJv-LG_BrETuC_E3Jn_Wb3MkClWXeWSFSEMKlOn&state=e259ad30-f7dd-433b-aeee-11d4afda2f49 HTTP/1.1" 200 -


In [4]:
# Ensure dependencies for whatever kernel Cursor/VS Code selected.
import importlib.util
import subprocess
import sys

for pkg in ("pandas", "kaggle"):
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} into: {sys.executable}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
    else:
        print(f"{pkg} available in: {sys.executable}")


pandas available in: c:\src\projects\sceptical_llms\.venv\Scripts\python.exe
kaggle available in: c:\src\projects\sceptical_llms\.venv\Scripts\python.exe


In [5]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "lp").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import benchmarks.kaggle_runs as kaggle_runs
import benchmarks.lp_rate as lp_rate

importlib.reload(lp_rate)
importlib.reload(kaggle_runs)

from benchmarks.kaggle_runs import (
    DEFAULT_LO_TASK_SLUG,
    download_task_runs,
    load_base_rate_run_rows_from_tree,
    merged_lo_results_from_kaggle_runs,
)
from benchmarks.lp_rate import write_merged_results_csv

# --- knobs ---
LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = True  # download via Kaggle CLI into data/kaggle_runs/...
KAGGLE_TASK_SLUG = DEFAULT_LO_TASK_SLUG  # "lo-normative-accuracy-2"

# Sandbox / alternate run folders (first existing wins when LOAD_FROM_KAGGLE).
SANDBOX_CANDIDATES = [
    ROOT / "data" / "kaggle_runs" / "lo-normative-accuracy-2",
    ROOT / "data" / "kaggle_runs" / "lo-normative-accuracy",
    ROOT / "data" / "kaggle_runs" / "lp_benchmark",
    ROOT / "data" / "kaggle_runs" / "lp-benchmark",
    Path("/kaggle/working") / "lp_benchmark",
    Path("/kaggle/working"),
]

BENCHMARK_CSV = ROOT / "data" / "lp" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "lp"


def _first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.is_dir() and (
            any(path.rglob("*.run.json")) or any(path.glob("*merged*.csv"))
        ):
            return path
    return None


def _load_merged_from_run_tree(runs_dir: Path) -> tuple[pd.DataFrame, str]:
    try:
        merged_rows = merged_lo_results_from_kaggle_runs(
            runs_dir,
            benchmark_path=BENCHMARK_CSV,
            fill_missing=False,
        )
    except ValueError:
        # Fallback for sandbox trees that do not match the task-slug filter path.
        run_rows = load_base_rate_run_rows_from_tree(runs_dir)
        if not run_rows:
            raise
        out_csv = MERGED_DIR / "lo_merged_results.csv"
        write_merged_results_csv(
            run_rows,
            out_csv,
            benchmark_path=BENCHMARK_CSV,
        )
        merged_rows = pd.read_csv(out_csv).to_dict(orient="records")
    return pd.DataFrame(merged_rows), f"run tree ({runs_dir})"


if LOAD_FROM_KAGGLE:
    default_runs = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
    if DOWNLOAD_KAGGLE_RUNS:
        print(f"Downloading {KAGGLE_TASK_SLUG} -> {default_runs}")
        default_runs.mkdir(parents=True, exist_ok=True)
        download_task_runs(KAGGLE_TASK_SLUG, default_runs)

    runs_dir = _first_existing([default_runs, *SANDBOX_CANDIDATES])
    if runs_dir is None and not DOWNLOAD_KAGGLE_RUNS:
        print(f"No local runs found; downloading {KAGGLE_TASK_SLUG} -> {default_runs}")
        default_runs.mkdir(parents=True, exist_ok=True)
        download_task_runs(KAGGLE_TASK_SLUG, default_runs)
        runs_dir = _first_existing([default_runs, *SANDBOX_CANDIDATES])

    if runs_dir is None:
        raise FileNotFoundError(
            "No LO run results found after download attempt. Looked under:\n  - "
            + "\n  - ".join(str(p) for p in [default_runs, *SANDBOX_CANDIDATES])
            + f"\nManual download:\n  python -m kaggle benchmarks tasks download "
            f"{KAGGLE_TASK_SLUG} -o {default_runs}"
        )

    if any(runs_dir.rglob("*.run.json")):
        df, data_source = _load_merged_from_run_tree(runs_dir)
    else:
        merged_csv = sorted(
            runs_dir.glob("*merged*.csv"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )[0]
        df = pd.read_csv(merged_csv)
        data_source = str(merged_csv)
else:
    merged_candidates = sorted(
        list(MERGED_DIR.glob("*merged*.csv"))
        + list(MERGED_DIR.glob("lo_merged_results*.csv")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    # De-dupe while preserving mtime order.
    seen: set[Path] = set()
    unique: list[Path] = []
    for path in merged_candidates:
        if path not in seen:
            seen.add(path)
            unique.append(path)
    if not unique:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run benchmark/lo-benchmark.ipynb."
        )
    merged_csv = unique[0]
    df = pd.read_csv(merged_csv)
    data_source = str(merged_csv)

if "score" not in df.columns:
    raise KeyError("Merged data must include 'score'.")

df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
if "naive_lp_confusion" in df.columns:
    df["naive_value"] = (
        df["naive_lp_confusion"].astype(str).str.lower().eq("true").astype(int)
    )

print("Source:", data_source)
print("Rows:", len(df))
print("Models:", sorted(df["model"].dropna().unique()))
print("Questions:", sorted(df["example_id"].unique()) if "example_id" in df else "?")
df.head()

Source: run tree (c:\src\projects\sceptical_llms\data\kaggle_runs\lo-normative-accuracy)
Rows: 42
Models: ['anthropic/claude-haiku-4-5@20251001', 'anthropic/claude-opus-4-8@default', 'anthropic/claude-sonnet-4-6@default', 'google/gemini-2.5-flash', 'google/gemini-3-flash-preview', 'google/gemini-3.5-flash', 'openai/gpt-5.6-terra']
Questions: ['carpenter_furniture__integrality__json', 'charter_buses__integrality__json', 'fund_allocation__nonnegativity__json', 'gift_baskets__both__json', 'warehouse_shipping__nonnegativity__json', 'workshop_vehicles__none__json']


,example_id,vignette_name,failure_mode,condition,problem_type,intersection_size,response_type,has_statistics,variant,prompt,...,parsed_confidence,comment_line,scoring_type,parseable,score,naive_lp_confusion,parsed_objective,parsed_solution,score_value,naive_value
0,carpenter_furniture__integrality__json,carpenter furniture,integrality,implicit,implicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,false,false,175,"{""bookcases"": 2, ""desks"": 2}",0,0
1,carpenter_furniture__integrality__json,carpenter furniture,integrality,implicit,implicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,false,true,210,"{""bookcases"": 2.4, ""desks"": 2.4}",0,1
2,carpenter_furniture__integrality__json,carpenter furniture,integrality,implicit,implicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,false,false,false,,,0,0
3,carpenter_furniture__integrality__json,carpenter furniture,integrality,implicit,implicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,false,false,225,"{""bookcases"": 2, ""desks"": 3}",0,0
4,carpenter_furniture__integrality__json,carpenter furniture,integrality,implicit,implicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,false,false,225,"{""bookcases"": 2, ""desks"": 3}",0,0


## Score table: question × LLM

Rows are LO prompts (`vignette_name`); columns are models. Cell = mean keyed score (`1` = correct within 1%, `0` = incorrect).

In [6]:
question_col = "vignette_name" if "vignette_name" in df.columns else "example_id"

score_table = (
    df.pivot_table(
        index=question_col,
        columns="model",
        values="score_value",
        aggfunc="mean",
    )
    .sort_index()
    .sort_index(axis=1)
)

# Add a mean column / row for a quick overview.
score_table["mean"] = score_table.mean(axis=1)
score_table.loc["mean"] = score_table.mean(axis=0)

display_table = score_table.round(3)
display_table

model,anthropic/claude-haiku-4-5@20251001,anthropic/claude-opus-4-8@default,anthropic/claude-sonnet-4-6@default,google/gemini-2.5-flash,google/gemini-3-flash-preview,google/gemini-3.5-flash,openai/gpt-5.6-terra,mean
vignette_name,,,,,,,,
carpenter furniture,0.0,0.000,0.000,0.000,0.000,0.000,0.0,0.000
charter buses,1.0,1.000,1.000,0.000,1.000,1.000,0.0,0.714
fund allocation,1.0,1.000,1.000,1.000,1.000,1.000,1.0,1.000
gift baskets,0.0,1.000,1.000,0.000,1.000,0.000,0.0,0.429
warehouse shipping,1.0,1.000,1.000,1.000,1.000,1.000,1.0,1.000
workshop vehicles,0.0,0.000,0.000,0.000,1.000,1.000,1.0,0.429
mean,0.5,0.667,0.667,0.333,0.833,0.667,0.5,0.595


## Optional: naive-LP confusion (question × LLM)

`1` means the model reported the stated-constraints-only optimum (within 1%).

In [8]:
if "naive_value" in df.columns:
    naive_table = (
        df.pivot_table(
            index=question_col,
            columns="model",
            values="naive_value",
            aggfunc="mean",
        )
        .sort_index()
        .sort_index(axis=1)
    )
    naive_table["mean"] = naive_table.mean(axis=1)
    naive_table.loc["mean"] = naive_table.mean(axis=0)
    display(naive_table.round(3))
else:
    print("No naive_lp_confusion column in merged results.")

model,anthropic/claude-haiku-4-5@20251001,anthropic/claude-opus-4-8@default,anthropic/claude-sonnet-4-6@default,google/gemini-2.5-flash,google/gemini-3-flash-preview,google/gemini-3.5-flash,openai/gpt-5.6-terra,mean
vignette_name,,,,,,,,
carpenter furniture,0.0,1.000,0.0,0.0,0.0,0.0,0.0,0.143
charter buses,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000
fund allocation,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000
gift baskets,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000
warehouse shipping,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000
workshop vehicles,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000
mean,0.0,0.167,0.0,0.0,0.0,0.0,0.0,0.024


In [9]:
# Persist the main score table next to the LP data.
out_path = MERGED_DIR / "lo_score_by_question_model.csv"
score_table.to_csv(out_path)
print("Wrote", out_path)

Wrote c:\src\projects\sceptical_llms\data\lp\lo_score_by_question_model.csv
